In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data")

files = sorted(DATA_DIR.glob("PRICE_AND_DEMAND_2025??_QLD1.csv"))

print(f"Found {len(files)} files:")
for file in files:
    print(file.name)

Found 12 files:
PRICE_AND_DEMAND_202501_QLD1.csv
PRICE_AND_DEMAND_202502_QLD1.csv
PRICE_AND_DEMAND_202503_QLD1.csv
PRICE_AND_DEMAND_202504_QLD1.csv
PRICE_AND_DEMAND_202505_QLD1.csv
PRICE_AND_DEMAND_202506_QLD1.csv
PRICE_AND_DEMAND_202507_QLD1.csv
PRICE_AND_DEMAND_202508_QLD1.csv
PRICE_AND_DEMAND_202509_QLD1.csv
PRICE_AND_DEMAND_202510_QLD1.csv
PRICE_AND_DEMAND_202511_QLD1.csv
PRICE_AND_DEMAND_202512_QLD1.csv


In [2]:
dfs = []

for file in files:
    df_month = pd.read_csv(file)
    dfs.append(df_month)

df = pd.concat(dfs, ignore_index=True)

print(df.shape)
df.head()


(105120, 5)


,REGION,SETTLEMENTDATE,TOTALDEMAND,RRP,PERIODTYPE
0,QLD1,2025/01/01 00:05:00,6444.99,154.30,TRADE
1,QLD1,2025/01/01 00:10:00,6469.55,150.42,TRADE
2,QLD1,2025/01/01 00:15:00,6410.01,150.07,TRADE
3,QLD1,2025/01/01 00:20:00,6344.94,132.58,TRADE
4,QLD1,2025/01/01 00:25:00,6354.61,132.08,TRADE


In [3]:
print(df.columns.tolist())
print(df.dtypes)
print(df.isna().sum())

print("\nRegions:")
print(df["REGION"].value_counts())

print("\nPeriod types:")
print(df["PERIODTYPE"].value_counts())

['REGION', 'SETTLEMENTDATE', 'TOTALDEMAND', 'RRP', 'PERIODTYPE']
REGION             object
SETTLEMENTDATE     object
TOTALDEMAND       float64
RRP               float64
PERIODTYPE         object
dtype: object
REGION            0
SETTLEMENTDATE    0
TOTALDEMAND       0
RRP               0
PERIODTYPE        0
dtype: int64

Regions:
REGION
QLD1    105120
Name: count, dtype: int64

Period types:
PERIODTYPE
TRADE    105120
Name: count, dtype: int64


In [4]:
df["SETTLEMENTDATE"] = pd.to_datetime(df["SETTLEMENTDATE"])

df = df.sort_values("SETTLEMENTDATE").reset_index(drop=True)

print(df["SETTLEMENTDATE"].min())
print(df["SETTLEMENTDATE"].max())
print(f"Total records: {len(df):,}")

2025-01-01 00:05:00
2026-01-01 00:00:00
Total records: 105,120


In [5]:
# Check duplicate timestamps
duplicates = df["SETTLEMENTDATE"].duplicated().sum()

# Check interval spacing
intervals = df["SETTLEMENTDATE"].diff().dropna()

print(f"Duplicate timestamps: {duplicates}")
print("\nInterval spacing:")
print(intervals.value_counts().head())


Duplicate timestamps: 0

Interval spacing:
SETTLEMENTDATE
0 days 00:05:00    105119
Name: count, dtype: int64


In [6]:
# Basic RRP statistics
print("2025 QLD RRP statistics ($/MWh)\n")

print(df["RRP"].describe())

# High-price percentiles
percentiles = [0.90, 0.95, 0.99, 0.995, 0.999]

print("\nHigh-price percentiles:")
for p in percentiles:
    value = df["RRP"].quantile(p)
    print(f"{p*100:5.1f}th percentile: ${value:,.2f}/MWh")

print(f"\nMinimum RRP: ${df['RRP'].min():,.2f}/MWh")
print(f"Maximum RRP: ${df['RRP'].max():,.2f}/MWh")

2025 QLD RRP statistics ($/MWh)

count    105120.000000
mean         84.889302
std         319.133141
min       -1000.000000
25%          34.750000
50%          78.730000
75%         110.820000
max       17304.000000
Name: RRP, dtype: float64

High-price percentiles:
 90.0th percentile: $155.54/MWh
 95.0th percentile: $199.28/MWh
 99.0th percentile: $285.15/MWh
 99.5th percentile: $332.27/MWh
 99.9th percentile: $1,033.21/MWh

Minimum RRP: $-1,000.00/MWh
Maximum RRP: $17,304.00/MWh


In [7]:
thresholds = [300, 500, 1000, 2000, 5000, 10000]

print("5-minute intervals above candidate thresholds:\n")

for threshold in thresholds:
    count = (df["RRP"] >= threshold).sum()
    percentage = count / len(df) * 100

    print(
        f">= ${threshold:>5,}/MWh: "
        f"{count:>5,} intervals "
        f"({percentage:.3f}%)"
    )

5-minute intervals above candidate thresholds:

>= $  300/MWh:   729 intervals (0.693%)
>= $  500/MWh:   222 intervals (0.211%)
>= $1,000/MWh:   148 intervals (0.141%)
>= $2,000/MWh:    91 intervals (0.087%)
>= $5,000/MWh:    72 intervals (0.068%)
>= $10,000/MWh:    44 intervals (0.042%)


In [9]:
# Add useful time fields
df["MONTH"] = df["SETTLEMENTDATE"].dt.month
df["HOUR"] = df["SETTLEMENTDATE"].dt.hour
df["DATE"] = df["SETTLEMENTDATE"].dt.date

threshold = 1000

high_price = df[df["RRP"] >= threshold].copy()

print(f"Intervals >= ${threshold:,}/MWh: {len(high_price)}")
print(f"Unique days containing them: {high_price['DATE'].nunique()}")

print("\nBy month:")
print(high_price.groupby("MONTH").size())

print("\nBy hour:")
print(high_price.groupby("HOUR").size())

print("\nDays with the most >= $1,000/MWh intervals:")
print(
    high_price.groupby("DATE")
    .size()
    .sort_values(ascending=False)
    .head(20)
)

Intervals >= $1,000/MWh: 148
Unique days containing them: 22

By month:
MONTH
1     29
3      3
4     24
5     13
6     78
10     1
dtype: int64

By hour:
HOUR
16     5
17    62
18    47
19    26
20     6
21     1
22     1
dtype: int64

Days with the most >= $1,000/MWh intervals:
DATE
2025-06-12    41
2025-01-22    25
2025-06-11    24
2025-04-07    14
2025-05-14     8
2025-06-26     7
2025-06-18     6
2025-04-09     6
2025-04-10     2
2025-05-13     2
2025-05-16     2
2025-05-18     1
2025-01-01     1
2025-04-12     1
2025-01-14     1
2025-04-04     1
2025-03-30     1
2025-03-21     1
2025-03-13     1
2025-01-28     1
dtype: int64


In [10]:
threshold = 1000

events_df = df[df["RRP"] >= threshold].copy()

# Time difference from previous qualifying interval
events_df["TIME_GAP"] = events_df["SETTLEMENTDATE"].diff()

# A new event starts whenever the previous qualifying interval
# was more than 5 minutes ago
events_df["NEW_EVENT"] = (
    events_df["TIME_GAP"].isna()
    | (events_df["TIME_GAP"] > pd.Timedelta(minutes=5))
)

events_df["EVENT_ID"] = events_df["NEW_EVENT"].cumsum()

events = (
    events_df.groupby("EVENT_ID")
    .agg(
        START=("SETTLEMENTDATE", "min"),
        END=("SETTLEMENTDATE", "max"),
        INTERVALS=("SETTLEMENTDATE", "size"),
        PEAK_RRP=("RRP", "max"),
        AVG_RRP=("RRP", "mean"),
        AVG_DEMAND=("TOTALDEMAND", "mean"),
        PEAK_DEMAND=("TOTALDEMAND", "max"),
    )
    .reset_index()
)

events["DURATION_MIN"] = events["INTERVALS"] * 5

print(f"Total events: {len(events)}")

print("\nEvent duration statistics:")
print(events["DURATION_MIN"].describe())

print("\nLongest events:")
print(
    events.sort_values("DURATION_MIN", ascending=False)
    .head(20)
    .to_string(index=False)
)

Total events: 40

Event duration statistics:
count     40.000000
mean      18.500000
std       35.140743
min        5.000000
25%        5.000000
50%        5.000000
75%       12.500000
max      190.000000
Name: DURATION_MIN, dtype: float64

Longest events:
 EVENT_ID               START                 END  INTERVALS  PEAK_RRP      AVG_RRP   AVG_DEMAND  PEAK_DEMAND  DURATION_MIN
       31 2025-06-12 17:05:00 2025-06-12 20:10:00         38  16567.63  9775.432895  8433.053158      8671.72           190
        5 2025-01-22 17:55:00 2025-01-22 19:40:00         22  14327.12 11804.085909 10802.659091     11119.26           110
       28 2025-06-11 17:15:00 2025-06-11 18:40:00         18  10478.01  4669.871111  8290.228333      8459.66            90
       14 2025-04-07 17:45:00 2025-04-07 18:10:00          6   1000.00  1000.000000  8067.053333      8101.58            30
       13 2025-04-07 17:10:00 2025-04-07 17:30:00          5  10207.99  4479.912000  7899.770000      7970.58            25

In [11]:
threshold = 1000
event_gap = pd.Timedelta(minutes=30)

events_df = df[df["RRP"] >= threshold].copy()

events_df["TIME_GAP"] = events_df["SETTLEMENTDATE"].diff()

events_df["NEW_EVENT"] = (
    events_df["TIME_GAP"].isna()
    | (events_df["TIME_GAP"] > event_gap)
)

events_df["EVENT_ID"] = events_df["NEW_EVENT"].cumsum()

events_30min = (
    events_df.groupby("EVENT_ID")
    .agg(
        START=("SETTLEMENTDATE", "min"),
        END=("SETTLEMENTDATE", "max"),
        QUALIFYING_INTERVALS=("SETTLEMENTDATE", "size"),
        PEAK_RRP=("RRP", "max"),
        AVG_QUALIFYING_RRP=("RRP", "mean"),
        AVG_DEMAND=("TOTALDEMAND", "mean"),
        PEAK_DEMAND=("TOTALDEMAND", "max"),
    )
    .reset_index()
)

# Elapsed time from first qualifying interval to final qualifying interval,
# including the final 5-minute interval
events_30min["EVENT_SPAN_MIN"] = (
    (events_30min["END"] - events_30min["START"]).dt.total_seconds() / 60
    + 5
)

print(f"Total 30-minute-gap events: {len(events_30min)}")

print("\nEvent span statistics:")
print(events_30min["EVENT_SPAN_MIN"].describe())

print("\nEvents:")
print(
    events_30min.sort_values("START")
    .to_string(index=False)
)

Total 30-minute-gap events: 27

Event span statistics:
count     27.000000
mean      32.962963
std       51.408232
min        5.000000
25%        5.000000
50%        5.000000
75%       35.000000
max      215.000000
Name: EVENT_SPAN_MIN, dtype: float64

Events:
 EVENT_ID               START                 END  QUALIFYING_INTERVALS  PEAK_RRP  AVG_QUALIFYING_RRP   AVG_DEMAND  PEAK_DEMAND  EVENT_SPAN_MIN
        1 2025-01-01 22:10:00 2025-01-01 22:10:00                     1  16220.18        16220.180000  6778.590000      6778.59             5.0
        2 2025-01-14 18:05:00 2025-01-14 18:05:00                     1   1049.38         1049.380000  9124.180000      9124.18             5.0
        3 2025-01-22 16:50:00 2025-01-22 16:55:00                     2  17304.00        13152.365000 10736.385000     10768.20            10.0
        4 2025-01-22 17:35:00 2025-01-22 19:40:00                    23  14327.12        11336.467826 10817.356957     11140.71           130.0
        5 2025-01-2

In [12]:
thresholds = [500, 1000, 2000]
gaps = [15, 30, 60]

results = []

for threshold in thresholds:
    qualifying = df[df["RRP"] >= threshold].copy()

    for gap_min in gaps:
        time_gap = qualifying["SETTLEMENTDATE"].diff()

        new_event = (
            time_gap.isna()
            | (time_gap > pd.Timedelta(minutes=gap_min))
        )

        event_ids = new_event.cumsum()

        grouped = qualifying.assign(EVENT_ID=event_ids).groupby("EVENT_ID")

        results.append({
            "THRESHOLD": threshold,
            "GAP_MIN": gap_min,
            "EVENTS": grouped.ngroups,
            "QUALIFYING_INTERVALS": len(qualifying),
            "UNIQUE_DAYS": qualifying["SETTLEMENTDATE"].dt.date.nunique()
        })

comparison = pd.DataFrame(results)

print(comparison.to_string(index=False))

 THRESHOLD  GAP_MIN  EVENTS  QUALIFYING_INTERVALS  UNIQUE_DAYS
       500       15      55                   222           41
       500       30      47                   222           41
       500       60      42                   222           41
      1000       15      33                   148           22
      1000       30      27                   148           22
      1000       60      23                   148           22
      2000       15      17                    91           13
      2000       30      16                    91           13
      2000       60      13                    91           13


In [13]:
thresholds = [500, 1000, 2000]

for threshold in thresholds:
    intervals = (df["RRP"] >= threshold).sum()
    hours = intervals * (5 / 60)

    print(
        f">= ${threshold:,}/MWh: "
        f"{intervals} intervals = {hours:.2f} hours"
    )
    

>= $500/MWh: 222 intervals = 18.50 hours
>= $1,000/MWh: 148 intervals = 12.33 hours
>= $2,000/MWh: 91 intervals = 7.58 hours
